# Day 7: PEFT / LoRA Dataset & Adapter Mechanics
### Learning Objectives & Success Criteria
- Transform domain incidents and ground-truth runbooks into Supervised Fine-Tuning (SFT) instruction pairs.
- Calculate Low-Rank Adaptation (`LoRA`, `r=8`) trainable parameter savings (`>98.9%`).
- Decide objectively between RAG, Prompt Engineering, and PEFT/LoRA.


### Cell 1 — Centralized Settings & Modular Engine Bootstrap


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from engine.settings import SETTINGS
from engine.router import ModelRouter
from engine.qdrant_store import QdrantVectorStore
from engine.use_case_loader import UseCaseLoader
from engine.visualizer import StoryVisualizer
from engine.debugger import TraceDebugger
from engine.quiz_helper import render_quiz

# Import PromptRegistry safely regardless of module filename (prompts.py or prompt_registry.py)
try:
    from engine.prompts import PromptRegistry
except ImportError:
    from engine.prompt_registry import PromptRegistry

# All URLs, paths, domain names, and cloud toggles come from SETTINGS (.env + configs/app_config.yaml)
# To test your own domain, see CUSTOM_USE_CASE_GUIDE.md and set ACTIVE_DOMAIN = "your_domain"
ACTIVE_DOMAIN = SETTINGS.active_domain

loader = UseCaseLoader(domain=ACTIVE_DOMAIN)
llm = ModelRouter(local_model_path=SETTINGS.local_model_path, enable_vertex=SETTINGS.enable_vertex_ai)
qdrant = QdrantVectorStore(collection_name=loader.collection_name, path=SETTINGS.qdrant_db_path)
prompts = PromptRegistry()
viz = StoryVisualizer()
dbg = TraceDebugger(debug_mode=False, enable_phoenix=False)

RUNBOOKS = loader.load_runbooks()
INCIDENTS = loader.load_incidents()
qdrant.upsert_documents(RUNBOOKS)

print(f"[Ready] Domain: {loader.domain} | LLM: {llm.provider} | VectorDB: {qdrant.mode} ({loader.collection_name})")


### Cell 2 — Production Incident Selector (0..9)


In [ ]:
# Select any of the 10 production incidents in the active domain (0..9)
INCIDENT_INDEX = 0
ACTIVE_INCIDENT = INCIDENTS[INCIDENT_INDEX]

print(f"Selected Incident: [{ACTIVE_INCIDENT['incident_id']}] {ACTIVE_INCIDENT['title']} (Severity: {ACTIVE_INCIDENT['severity']})")
print("Raw Log:", ACTIVE_INCIDENT["raw_log"])


### Cell 3 — Engineering Concept Brief
**Concept: Low-Rank Adaptation (LoRA)**
Builds SFT instruction-JSON training pairs from the active domain's 10 incidents and computes rank-8 parameter reduction.


### Cell 4 — Traced Execution & X-Ray Visual Storyboard


In [ ]:
dbg.start_run("day07_lora")
with dbg.start_span(f"Day7_PEFT_LoRA_{loader.domain}", span_kind="CHAIN") as root_span:
    t0 = time.perf_counter()
    sft_pairs = [
        {
            "instruction": f"Triage {loader.domain} incident and identify severity + runbook.",
            "input": inc["raw_log"],
            "output": json.dumps({"incident_id": inc["incident_id"], "severity": inc["severity"], "runbook": inc["expected_runbook"]})
        }
        for inc in INCIDENTS
    ]
    t_ds = round((time.perf_counter() - t0) * 1000, 2)
    dbg.record_step("build_sft_dataset", {"domain": loader.domain}, {"num_pairs": len(sft_pairs), "sample": sft_pairs[0]}, t_ds, span_kind="CHAIN")

    hidden_size, lora_rank = 1536, 8
    savings_pct = round((1 - (2 * hidden_size * lora_rank) / (hidden_size * hidden_size)) * 100, 2)
    math_summary = {"hidden_size": hidden_size, "rank": lora_rank, "parameter_savings_pct": savings_pct}
    dbg.record_step("compute_lora_math", {"rank": lora_rank}, math_summary, 0.1, span_kind="CHAIN")

viz.render_story(
    title=f"Day 7 LoRA SFT Dataset — {loader.domain}",
    stages=[
        {"name": "Curated SFT Pairs", "badge": "DATASET", "latency_ms": t_ds, "summary": f"{len(sft_pairs)} pairs built dynamically", "payload": sft_pairs[:2]},
        {"name": "LoRA Adapter Efficiency", "badge": "PEFT MATH", "latency_ms": 0.1, "summary": f"{savings_pct}% fewer trainable parameters", "payload": math_summary}
    ],
    phoenix_url=dbg.phoenix_url
)


### Cell 5 — Guarded Cloud Parity Check (`SETTINGS.enable_vertex_ai`)


In [ ]:
if SETTINGS.enable_vertex_ai:
    print("Vertex AI Supervised Fine-Tuning pipeline active.")
else:
    print("ENABLE_VERTEX_AI=false -> Local LoRA dataset prep verified.")


### Cell 6 — Automated Validation Gate


In [ ]:
assert len(sft_pairs) == 10 and savings_pct > 98.0
print("[PASS] Day 7 PEFT / LoRA Gate passed!")


### Cell 7 — Active Lab Experiment
**Hands-On Lab Challenge:** Change `lora_rank = 16` or `32` in Cell 4 and re-run to compare the trainable parameter savings percentage against `rank = 8`.


### Cell 8 — Interactive Self-Assessment Questionnaire


In [ ]:
import json, os, sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from engine.quiz_helper import render_quiz

with open(repo_root / "configs" / "quizzes" / "day07.json", "r", encoding="utf-8") as f:
    quiz_cfg = json.load(f)

render_quiz(quiz_cfg["title"], quiz_cfg["questions"])


### Cell 9 — Bring Your Own Use Case (BYOUC)
Want to run all 10 notebooks against your own enterprise domain? Follow **[CUSTOM_USE_CASE_GUIDE.md](../CUSTOM_USE_CASE_GUIDE.md)** (`python scripts/scaffold_domain.py --domain your_domain --prefix YOUR-`) and set `ACTIVE_DOMAIN = "your_domain"` in Cell 1.
